# Suite EVL — Evaluators

`Evaluators.OfAny(term, scope)` computes a term's value with the variables in `scope` bound, and `Evaluators.OfLiteral`,
`OfCall`, `OfVariable`, `OfLet` and `OfQuantifier` evaluate one kind. A term is a value of
`Expressions.Framework.Term`, built in a store, in a transaction (here through `term`, from a writer). Evaluation is
three-valued (absent and incomparable values are unknown, and logic follows Kleene), never coerces native types, and
evaluates only the core operations.

In [ ]:
import math

from mbse.Expressions import Framework
from mbse.Expressions.Dialects import Basic
from mbse.Expressions.Dialects.Basic import Evaluators as V, Expressions as E
from mbse.Expressions.Framework import Terms as F
from mbse.Schemas.Framework import Builders
from support import raises

store = Builders.Store()
store.transaction(Framework.install)
store.transaction(Basic.install)


def term(spec):
    """The term value `spec` writes, built in a transaction."""
    with store.transaction():
        return store.Expressions.Framework.Term(E.DIALECT.spec(spec))


def ev(spec, **scope):
    """Evaluates the term `spec` writes, with `scope` bound."""
    return V.OfAny(term(spec), scope)


def people(txn):
    """A person, a reference object; a card that reaches its holder by phone or email, a union of value objects."""
    S = txn.Schemas
    return txn.uid("evl.people/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Person": S.OfObject, "Reach": S.OfUnion, "Code": S.OfUnion, "Card": S.OfObject})
        .spec([
            lambda _: _.Person.name("evl.Person").ref(True).properties({"name": S.OfStr, "age": S.OfInt}),
            lambda _: _.Reach.name("evl.Reach").branches([
                lambda t: t.name("phone").schema([S.OfObject(lambda o: o.properties({"kind": S.OfStr, "number": S.OfStr}))]),
                lambda t: t.name("email").schema([S.OfObject(lambda o: o.properties({"kind": S.OfStr, "address": S.OfStr}))]),
            ]),
            lambda _: _.Code.name("evl.Code").branches([  # a union of scalars
                lambda t: t.name("number").schema([S.OfInt]), lambda t: t.name("text").schema([S.OfStr]),
                lambda t: t.name("digits").schema([S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfInt]))])]),
            lambda _: _.Card.name("evl.Card").ref(True).properties({"reach": _.Reach, "code": _.Code}),
        ])]))])


store.transaction(people)

## EVL-01 · Evaluating literals, variables, lets, `get` and `has`

In [ ]:
with store.transaction():
    ann = store.evl.Person(lambda p: p.name("Ann").age(30))
    nobody = store.evl.Person(lambda p: p)
person = E.variable("person")
assert ev(3) == 3 and ev(E.literal("x")) == "x" and ev(person, person=ann) is ann
assert ev(person.age, person=ann) == 30 and ev(person.age, person=nobody) is None  # absent: unknown
assert ev(person.has("age"), person=ann) is True and ev(person.has("age"), person=nobody) is False
assert ev(person.age, person=None) is None and ev(person.has("age"), person=None) is None  # unknown object
assert ev(E.let_("a", person.age, E.variable("a").add(1)), person=ann) == 31
assert ev(E.let_("a", 1, E.let_("a", 2, E.variable("a")))) == 2  # the innermost binding wins
assert ev(E.let_("a", 1, E.variable("a").add(E.let_("a", 2, E.variable("a"))))) == 3  # and only in its body
one, term_value = term(E.literal(1)), E.variable("term_value")
assert ev(term_value.has("literal"), term_value=one) is True and ev(term_value.literal.value, term_value=one) == 1  # terms are values too
with raises(KeyError, match="variable 'x' is not bound"):
    ev(E.variable("x"))
with raises(TypeError, match="get expects an object, got int"):
    ev(E.operation("get", 1, "x"))
with raises(TypeError, match="has expects a property name, got int"):
    ev(E.operation("has", person, 1), person=ann)

## EVL-02 · Comparisons are three-valued and never coerce

In [ ]:
unit = E.literal(1)
assert [ev(unit.lt(2)), ev(unit.le(1)), ev(unit.gt(2)), ev(unit.ge(1)), ev(unit.eq(1)), ev(unit.ne(1))] == [
    True, True, False, True, True, False]
assert ev(E.literal("b").gt("a")) is True and ev(E.literal(b"\x01").lt(b"\x02")) is True
assert ev(unit.eq(1.0)) is None and ev(unit.lt(1.5)) is None and ev(unit.ne("1")) is None  # different types
assert ev(E.literal(True).eq(False)) is False and ev(E.literal(True).lt(False)) is None  # bools are not ordered
assert ev(E.literal(True).le(True)) is True
nan = E.literal(math.nan)
assert ev(nan.eq(math.nan)) is True and ev(nan.eq(1.0)) is False and ev(nan.lt(1.0)) is None
assert ev(E.literal(-0.0).lt(0.0)) is True and ev(E.literal(-0.0).eq(0.0)) is False
age = E.variable("x").age
assert ev(age.eq(1), x=nobody) is None and ev(age.ne(1), x=nobody) is None and ev(age.lt(1), x=nobody) is None
first, second = E.variable("a"), E.variable("b")
assert ev(first.eq(second), a=ann, b=ann) is True and ev(first.eq(second), a=ann, b=nobody) is False  # objects: by identity
assert ev(first.lt(second), a=ann, b=nobody) is None and ev(first.eq(1), a=ann) is None

## EVL-03 · Logic follows Kleene; arithmetic keeps one type

In [ ]:
T, F_, U = E.literal(True), E.literal(False), E.variable("u")  # u is unknown


def table(method):
    return [[ev(getattr(x, method)(y), u=None) for y in (T, F_, U)] for x in (T, F_, U)]


assert table("and_") == [[True, False, None], [False, False, False], [None, False, None]]
assert table("or_") == [[True, True, True], [True, False, None], [True, None, None]]
assert table("implies") == [[True, False, None], [True, True, True], [True, None, None]]
assert [ev(x.not_(), u=None) for x in (T, F_, U)] == [False, True, None]
missing = E.variable("missing")  # never evaluated when the first operand decides
assert ev(F_.and_(missing)) is False and ev(T.or_(missing)) is True and ev(F_.implies(missing)) is True
with raises(TypeError, match="and expects bool operands, got int"):
    ev(T.and_(1))
with raises(TypeError, match="not expects bool operands, got str"):
    ev(E.literal("x").not_())
pair = E.literal(2)
assert [ev(pair.add(3)), ev(pair.sub(3)), ev(pair.mul(3)), ev(pair.neg())] == [5, -1, 6, -2]
assert ev(E.literal(1.5).add(1.0)) == 2.5 and ev(E.literal(2**70).mul(2)) == 2**71
assert ev(U.add(1), u=None) is None and ev(U.neg(), u=None) is None
with raises(TypeError, match="add expects numbers of one domain, got int and float"):
    ev(E.literal(1).add(1.5))
with raises(TypeError, match="sub expects numbers of one domain, got bool and bool"):
    ev(T.sub(T))
with raises(TypeError, match="neg expects a number, got str"):
    ev(E.literal("x").neg())

## EVL-04 · Checking: `validate()` finds what evaluation would raise

In [ ]:
assert E.DIALECT.validate(term(E.operation("lt", 1))) == ["lt takes 2 arguments, got 1"]
foo = term(E.operation("foo", 1))
assert E.DIALECT.validate(foo) == E.DIALECT.validate(foo, core=True) == ["'foo' is not an operation Basic defines"]
person = E.variable("person")
checked = term(E.let_("a", person.age, E.variable("a").ge(18).and_(person.has("email"))))
assert E.DIALECT.validate(checked, bound={"person"}, core=True) == []
assert E.DIALECT.validate(checked) == ["value: argument 0: variable 'person' is not bound",
                                       "body: argument 1: argument 0: variable 'person' is not bound"]
with raises(NotImplementedError, match="'foo' is not an operation Basic defines"):
    V.OfAny(foo)
with raises(TypeError, match="lt takes 2 arguments, got 1"):
    ev(E.operation("lt", 1))
with raises(ValueError, match="a literal needs a value"):
    ev(E.literal(None))
with raises(ValueError, match="a let needs a value"):
    ev(E.let_("x", None, None))
with raises(ValueError, match="a let needs a body"):
    ev(E.let_("x", 1, None))
# A term is a nested value: written into another, it is copied, so a term never holds itself.
inner = term(E.operation("not", True))
outer = term(E.operation("not", inner))
assert ev(outer) is True and len(list(F.walk(outer))) == 3 and len(list(F.walk(inner))) == 2
with raises(TypeError, match="expected a term, a writer or a native value, got [7]"):
    E.operation("not", [7])

## EVL-05 · The union of terms names each branch by its kind's tag

In [ ]:
Term = store.registered("Expressions.Framework.Term")
assert [branch.name for branch in Term.branches] == ["literal", "call", "variable", "let", "list", "mapping", "quantifier"]
assert [branch.schema.name for branch in Term.branches] == [
    "Expressions.Framework.Literal", "Expressions.Framework.Call", "Expressions.Framework.Variable",
    "Expressions.Framework.Let", "Expressions.Framework.List", "Expressions.Framework.Mapping",
    "Expressions.Basic.Quantifier"]
samples = [E.literal(1), E.operation("f", 1, "x"), E.variable("x"), E.let_("a", 1, 2), E.list_(1), E.mapping(("k", 1)),
           E.quantifier("all", "x", E.variable("xs"), True)]
assert [F.kind_of(term(sample))[0] for sample in samples] == [branch.name for branch in Term.branches]
assert Term.validate() == []

## EVL-06 · Each kind has its own entry point, which takes a term of that kind

In [ ]:
x = E.variable("x")
assert V.OfLiteral(term(3)) == 3 and V.OfLiteral(term(E.literal("s"))) == "s"
assert V.OfVariable(term(x), {"x": 2}) == 2
assert V.OfCall(term(x.add(1)), {"x": 1}) == 2 and V.OfLet(term(E.let_("y", 1, E.variable("y").neg()))) == -1
assert V.OfQuantifier(term(x.all("i", E.variable("i").gt(0))), {"x": [1, 2]}) is True
assert V.OfAny(term(x), {"x": None}) is None
with store.transaction():
    expression = store.Expressions.Framework.Expression(lambda e: e.root(x.add(1)))
assert V.OfAny(expression, {"x": 1}) == 2 and V.OfCall(expression.root, {"x": 2}) == 3  # an expression, its root
with raises(TypeError, match="expected a let, got a literal"):
    V.OfLet(term(3))
with raises(TypeError, match="expected a call, got a variable"):
    V.OfCall(term(x))  # a variable, not a call
with raises(TypeError, match="not a term: 3"):
    V.OfAny(3)

## EVL-07 · A constraint about a value, bound to a variable, reads value objects and union values

In [ ]:
card, mail = E.variable("card"), E.variable("mail")  # a card, and an email, a value object
with store.transaction():
    good = store.evl.Card(lambda c: c.reach(lambda r: r.email(lambda e: e.kind("work").address("a@x"))))
    phone = store.evl.Card(lambda c: c.reach(lambda r: r.phone(lambda e: e.number("1"))))
    same = store.evl.Card(lambda c: c.reach(lambda r: r.email(lambda e: e.kind("work").address("a@x"))))
    home = store.evl.Card(lambda c: c.reach(lambda r: r.email(lambda e: e.kind("home").address("a@x"))))
    work_phone = store.evl.Card(lambda c: c.reach(lambda r: r.phone(lambda e: e.kind("work"))))
email = card.get("reach").get("email")  # a union value is a record of its one branch
assert ev(card.get("reach").has("email"), card=good) is True and ev(card.get("reach").has("phone"), card=good) is False
assert ev(email.get("kind").eq("work"), card=good) is True
assert ev(email.get("kind").eq("home"), card=phone) is None
assert ev(mail.get("kind").eq("phone"), mail=good.reach.email) is False and ev(mail.has("kind"), mail=None) is None
assert ev(mail.get("kind"), mail=good.reach.email) == "work"  # a value object's property
assert ev(mail.eq(mail), mail=good.reach.email) is True  # a value object compares deeply, by its properties
other = E.variable("other")
reach = card.get("reach").eq(other.get("reach"))  # union values: the same branch, equal
assert ev(reach, card=good, other=same) is True
assert ev(reach, card=good, other=home) is False
assert ev(reach, card=good, other=work_phone) is None  # another branch
assert ev(card.eq(other), card=good, other=good.reach.email) is None and ev(card.eq(other), card=good, other=good) is True
with store.transaction():
    coded = store.evl.Card(lambda c: c.code(lambda r: r.number(7)))
assert ev(card.get("code").get("number"), card=coded) == 7 and ev(card.get("code").has("text"), card=coded) is False
with store.transaction():
    listed = store.evl.Card(lambda c: c.code(lambda r: r.digits(lambda k: (k.at(0).as_scalar(1), k.at(1).as_scalar(2), k)[2])))
assert ev(card.get("code").get("digits").count(), card=listed) == 2  # a union's container branch reads as a collection